# 14A — ONE-TIME Official Test Evaluation — Frozen Final Pipeline

**Project:** Pediatric Chest X-ray / final sealed evaluation  
**Controller ID:** `T01_OneTime_Official_Test_Evaluation`  
**Dataset:** VinDr-PCXR official test set  
**Official test size:** expected **1,397 images**  
**Platform:** Kaggle  
**Target GPU:** NVIDIA T4  

# ⚠️ Irreversible research step

This notebook is the first notebook allowed to read:

- `image_labels_test.csv`
- `annotations_test.csv`
- the official `test/` DICOM folder

The project plan explicitly kept the official test set sealed until architecture, losses, calibration, abstention policy, ablations, and repeated-seed experiments were frozen. VinDr-PCXR provides separate official test image labels and test bounding-box annotations, and those labels must not be used for model selection or calibration.  

**After this notebook is run with test authorization enabled:**

> **Do not tune, retrain, reselect, recalibrate, or redesign any component using the observed test results.**

If a test result is disappointing, report it honestly.

---

## Frozen pipeline being evaluated

### Disease classification robustness — 3 independent seeds

- Seed 42: `M01_DiseaseFinding_Multitask_Seed42`
- Seed 123: `F01_Multitask_Seed123`
- Seed 2026: `F01_Multitask_Seed2026`

These are evaluated independently.  
**No post-hoc ensemble is introduced.**

### Radiologist-box localization robustness — 3 independent seeds

- Seed 42: `M08_DecoupledLocalizationAdapter_Seed42`
- Seed 123: `F02_LocalizationAdapter_Seed123`
- Seed 2026: `F02_LocalizationAdapter_Seed2026`

Each adapter is paired only with its matching frozen classifier.

### Calibration — seed-42 trustworthiness pipeline only

Frozen 12A scaler:

- `M13_GlobalTemperatureScaling_Seed42`
- one global scalar temperature
- fitted using the held-out calibration split only

The same frozen temperature is applied to seed-42 official-test logits without refitting.

### Selective prediction — seed-42 trustworthiness pipeline only

Two already-frozen development policies are evaluated:

1. `M14_CalibratedSelectivePrediction_Seed42`
   - naive calibrated mean-entropy abstention
2. `M15_EvidenceProtectedSelectivePrediction_Seed42`
   - evidence-protected calibrated entropy policy

Their thresholds are applied exactly as frozen.  
No test-derived coverage threshold is allowed.

---

## Final test outputs

### Classification
- per-seed 14-pathology Macro-AUPRC
- per-seed Macro-AUROC
- Macro-F1 @ 0.5
- Head / Middle / Tail Macro-AUPRC
- 3-seed mean ± SD
- per-class 3-seed mean ± SD

### Localization
- direct-CAM attention mass
- adapter attention mass
- pointing-game accuracy
- direct-CAM → adapter gain
- 3-seed mean ± SD
- per-finding localization stability

### Calibration
Seed-42 only:
- NLL / BCE before vs frozen temperature scaling
- Brier score before vs after
- ECE before vs after
- ranking invariance check

### Selective prediction
Seed-42 only:
- achieved official-test coverage
- Hamming risk
- Brier risk
- Macro-AUPRC / AUROC / F1 on retained cases
- Head / Middle / Tail positive rejection
- tail-positive rejection
- M14 naive vs M15 evidence-protected comparison
- risk–coverage AURC using the frozen uncertainty definition

---

## Restricted-data export policy

The output ZIP contains **aggregate and per-class metrics only**.

It deliberately does **not** export:

- DICOM files
- X-ray images
- raw bounding-box coordinates
- per-image test IDs
- per-image official-test labels
- per-image official-test predictions

This keeps restricted official-test records out of the portable result ZIP.


In [1]:
# ============================================================
# 1. Imports + FINAL TEST AUTHORIZATION SWITCH
# ============================================================
import os
import sys
import gc
import io
import json
import time
import math
import random
import hashlib
import zipfile
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pydicom
from pydicom.pixels import apply_modality_lut, apply_voi_lut
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode
from torchvision.models import resnet50

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    recall_score,
    confusion_matrix,
)

from tqdm.auto import tqdm

# ------------------------------------------------------------------
# IMPORTANT:
# Keep False while only checking attached frozen artifacts.
# Change to True exactly once when you intentionally open the
# official test labels and run the final evaluation.
# ------------------------------------------------------------------
AUTHORIZE_OFFICIAL_TEST = False

CONTROLLER_ID = "T01_OneTime_Official_Test_Evaluation"

EXPECTED_SPLIT_MANIFEST_SHA256 = (
    "3d2f8de854c9f61382b1ef5594264af739fcb1d3d9ca8ca9fe8d388380363770"
)

EXPECTED_TEST_IMAGES = 1397

IMAGE_SIZE = 320
BOX_MASK_SIZE = 80
BATCH_SIZE = 8
NUM_WORKERS = 4
FIXED_THRESHOLD = 0.5

SEEDS = [
    42,
    123,
    2026,
]

BASE_EXPERIMENT_IDS = {
    42:
        "M01_DiseaseFinding_Multitask_Seed42",
    123:
        "F01_Multitask_Seed123",
    2026:
        "F01_Multitask_Seed2026",
}

ADAPTER_EXPERIMENT_IDS = {
    42:
        "M08_DecoupledLocalizationAdapter_Seed42",
    123:
        "F02_LocalizationAdapter_Seed123",
    2026:
        "F02_LocalizationAdapter_Seed2026",
}

CALIBRATION_EXPERIMENT_ID = (
    "M13_GlobalTemperatureScaling_Seed42"
)

NAIVE_SELECTIVE_EXPERIMENT_ID = (
    "M14_CalibratedSelectivePrediction_Seed42"
)

PROTECTED_SELECTIVE_EXPERIMENT_ID = (
    "M15_EvidenceProtectedSelectivePrediction_Seed42"
)

OUTPUT_DIR = Path(
    f"/kaggle/working/PediCGAM/{CONTROLLER_ID}"
)

TABLE_DIR = (
    OUTPUT_DIR
    / "tables"
)

FIG_DIR = (
    OUTPUT_DIR
    / "figures"
)

CACHE_DIR = Path(
    f"/kaggle/working/PediCGAM_cache/{CONTROLLER_ID}_{IMAGE_SIZE}"
)

EXTRACT_DIR = Path(
    "/kaggle/working/14a_extracted"
)

for d in [
    OUTPUT_DIR,
    TABLE_DIR,
    FIG_DIR,
    CACHE_DIR,
    EXTRACT_DIR,
]:
    d.mkdir(
        parents=True,
        exist_ok=True,
    )

KAGGLE_INPUT = Path(
    "/kaggle/input"
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

if (
    DEVICE.type
    != "cuda"
):
    raise RuntimeError(
        "GPU is not active. Enable a Kaggle GPU accelerator."
    )

print(
    "Notebook build:",
    "14A_ONE_TIME_OFFICIAL_TEST_V2_CLEAN_GATE",
)
print(
    "Controller:",
    CONTROLLER_ID,
)
print(
    "GPU:",
    torch.cuda.get_device_name(
        0
    ),
)
print(
    "Official-test authorization:",
    AUTHORIZE_OFFICIAL_TEST,
)


Notebook build: 14A_ONE_TIME_OFFICIAL_TEST_V2_CLEAN_GATE
Controller: T01_OneTime_Official_Test_Evaluation
GPU: Tesla T4
Official-test authorization: False


# 2. Required Kaggle inputs

Attach these **9 inputs** before enabling the official-test switch:

1. **VinDr-PCXR**
2. **Stage-02 frozen split output**
3. **09A / M01 seed-42 output**
4. **10E / M08 seed-42 localization output**
5. **13A / F01 3-seed multitask output**
6. **13B / F02 3-seed localization output**
7. **12A / M13 calibration output**
8. **12B / M14 naive selective-prediction output**
9. **12C / M15 evidence-protected selective-prediction output**

The notebook verifies all frozen artifacts **before** the authorization gate permits access to official test labels.


In [2]:
# ============================================================
# 3. Generic direct-file / ZIP artifact helpers
# ============================================================
def load_torch_safely(
    path,
):
    return torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )


def find_torch_checkpoint(
    filename,
    experiment_id,
):
    # Direct files first
    for p in KAGGLE_INPUT.rglob(
        filename
    ):
        try:
            ckpt = load_torch_safely(
                p
            )

            if (
                ckpt.get(
                    "experiment_id"
                )
                == experiment_id
            ):
                return (
                    p,
                    "direct",
                    ckpt,
                )
        except Exception:
            pass

    # Then ZIPs
    for zip_path in KAGGLE_INPUT.rglob(
        "*.zip"
    ):
        try:
            with zipfile.ZipFile(
                zip_path,
                "r",
            ) as zf:
                members = [
                    n
                    for n in zf.namelist()
                    if n.endswith(
                        "/"
                        + filename
                    )
                    or n
                    == filename
                ]

                for member in members:
                    target = (
                        EXTRACT_DIR
                        / (
                            experiment_id
                            + "_"
                            + filename
                        )
                    )

                    target.write_bytes(
                        zf.read(
                            member
                        )
                    )

                    try:
                        ckpt = load_torch_safely(
                            target
                        )

                        if (
                            ckpt.get(
                                "experiment_id"
                            )
                            == experiment_id
                        ):
                            return (
                                target,
                                f"zip:{zip_path.name}",
                                ckpt,
                            )

                    except Exception:
                        pass

        except zipfile.BadZipFile:
            pass

    return (
        None,
        None,
        None,
    )


def find_json_by_experiment_id(
    filename,
    experiment_id,
):
    # Direct
    for p in KAGGLE_INPUT.rglob(
        filename
    ):
        try:
            with open(
                p,
                "r",
            ) as f:
                data = json.load(
                    f
                )

            if (
                data.get(
                    "experiment_id"
                )
                == experiment_id
            ):
                return (
                    p,
                    "direct",
                    data,
                )
        except Exception:
            pass

    # ZIP
    for zip_path in KAGGLE_INPUT.rglob(
        "*.zip"
    ):
        try:
            with zipfile.ZipFile(
                zip_path,
                "r",
            ) as zf:
                members = [
                    n
                    for n in zf.namelist()
                    if n.endswith(
                        "/"
                        + filename
                    )
                    or n
                    == filename
                ]

                for member in members:
                    try:
                        raw = zf.read(
                            member
                        )

                        data = json.loads(
                            raw.decode(
                                "utf-8"
                            )
                        )

                        if (
                            data.get(
                                "experiment_id"
                            )
                            == experiment_id
                        ):
                            target = (
                                EXTRACT_DIR
                                / (
                                    experiment_id
                                    + "_"
                                    + filename
                                )
                            )

                            target.write_bytes(
                                raw
                            )

                            return (
                                target,
                                f"zip:{zip_path.name}",
                                data,
                            )

                    except Exception:
                        pass

        except zipfile.BadZipFile:
            pass

    return (
        None,
        None,
        None,
    )


def locate_stage02_csv(
    filename,
):
    direct = list(
        KAGGLE_INPUT.rglob(
            filename
        )
    )

    if (
        direct
    ):
        return (
            direct[
                0
            ],
            "direct",
        )

    for zip_path in KAGGLE_INPUT.rglob(
        "*.zip"
    ):
        try:
            with zipfile.ZipFile(
                zip_path,
                "r",
            ) as zf:
                members = [
                    n
                    for n in zf.namelist()
                    if n.endswith(
                        filename
                    )
                ]

                if (
                    members
                ):
                    target = (
                        EXTRACT_DIR
                        / filename
                    )

                    target.write_bytes(
                        zf.read(
                            members[
                                0
                            ]
                        )
                    )

                    return (
                        target,
                        f"zip:{zip_path.name}",
                    )

        except zipfile.BadZipFile:
            pass

    return (
        None,
        None,
    )


In [3]:
# ============================================================
# 4. Verify all three frozen classifier checkpoints
# ============================================================
BASE_CHECKPOINTS = {}

for seed in SEEDS:
    experiment_id = (
        BASE_EXPERIMENT_IDS[
            seed
        ]
    )

    path, source, ckpt = (
        find_torch_checkpoint(
            "best_model.pth",
            experiment_id,
        )
    )

    if (
        path
        is None
    ):
        raise FileNotFoundError(
            "Missing frozen classifier checkpoint: "
            + experiment_id
        )

    if (
        ckpt.get(
            "split_manifest_sha256"
        )
        != EXPECTED_SPLIT_MANIFEST_SHA256
    ):
        raise RuntimeError(
            f"STOP: seed {seed} classifier split hash mismatch."
        )

    if (
        int(
            ckpt.get(
                "random_seed"
            )
        )
        != seed
    ):
        raise RuntimeError(
            f"STOP: seed {seed} classifier random_seed mismatch."
        )

    if (
        int(
            ckpt.get(
                "image_size"
            )
        )
        != IMAGE_SIZE
    ):
        raise RuntimeError(
            f"STOP: seed {seed} classifier image size mismatch."
        )

    if (
        not np.isclose(
            float(
                ckpt.get(
                    "finding_loss_weight"
                )
            ),
            0.5,
        )
    ):
        raise RuntimeError(
            f"STOP: seed {seed} classifier finding-loss weight mismatch."
        )

    if (
        ckpt.get(
            "boxes_used"
        )
        is not False
    ):
        raise RuntimeError(
            f"STOP: seed {seed} classifier unexpectedly used boxes."
        )

    BASE_CHECKPOINTS[
        seed
    ] = {
        "path":
            path,
        "source":
            source,
        "checkpoint":
            ckpt,
    }

    print(
        f"✅ Classifier seed {seed}:",
        experiment_id,
        "|",
        source,
    )


REFERENCE_LABEL_COLUMNS = (
    BASE_CHECKPOINTS[
        42
    ][
        "checkpoint"
    ][
        "disease_label_columns"
    ]
)

REFERENCE_FINDING_CLASSES = (
    BASE_CHECKPOINTS[
        42
    ][
        "checkpoint"
    ][
        "finding_classes"
    ]
)

if (
    len(
        REFERENCE_LABEL_COLUMNS
    )
    != 15
    or "No finding"
    not in REFERENCE_LABEL_COLUMNS
):
    raise RuntimeError(
        "STOP: frozen disease label order is invalid."
    )

if (
    len(
        REFERENCE_FINDING_CLASSES
    )
    != 37
    or "No finding"
    not in REFERENCE_FINDING_CLASSES
):
    raise RuntimeError(
        "STOP: frozen finding label order is invalid."
    )

for seed in [
    123,
    2026,
]:
    ckpt = (
        BASE_CHECKPOINTS[
            seed
        ][
            "checkpoint"
        ]
    )

    if (
        ckpt[
            "disease_label_columns"
        ]
        != REFERENCE_LABEL_COLUMNS
    ):
        raise RuntimeError(
            f"STOP: disease-label order mismatch for seed {seed}."
        )

    if (
        ckpt[
            "finding_classes"
        ]
        != REFERENCE_FINDING_CLASSES
    ):
        raise RuntimeError(
            f"STOP: finding-label order mismatch for seed {seed}."
        )

print(
    "✅ Disease/finding label order identical across all frozen classifiers."
)


✅ Classifier seed 42: M01_DiseaseFinding_Multitask_Seed42 | direct
✅ Classifier seed 123: F01_Multitask_Seed123 | direct
✅ Classifier seed 2026: F01_Multitask_Seed2026 | direct
✅ Disease/finding label order identical across all frozen classifiers.


In [4]:
# ============================================================
# 5. Verify all three frozen localization-adapter checkpoints
# ============================================================
ADAPTER_CHECKPOINTS = {}

for seed in SEEDS:
    experiment_id = (
        ADAPTER_EXPERIMENT_IDS[
            seed
        ]
    )

    path, source, ckpt = (
        find_torch_checkpoint(
            "best_localization_adapter.pth",
            experiment_id,
        )
    )

    if (
        path
        is None
    ):
        raise FileNotFoundError(
            "Missing frozen localization adapter: "
            + experiment_id
        )

    if (
        ckpt.get(
            "base_experiment_id"
        )
        != BASE_EXPERIMENT_IDS[
            seed
        ]
    ):
        raise RuntimeError(
            f"STOP: seed {seed} adapter/base pairing mismatch."
        )

    if (
        ckpt.get(
            "split_manifest_sha256"
        )
        != EXPECTED_SPLIT_MANIFEST_SHA256
    ):
        raise RuntimeError(
            f"STOP: seed {seed} adapter split hash mismatch."
        )

    if (
        int(
            ckpt.get(
                "random_seed",
                seed,
            )
        )
        != seed
    ):
        raise RuntimeError(
            f"STOP: seed {seed} adapter random_seed mismatch."
        )

    if (
        ckpt.get(
            "base_model_frozen"
        )
        is not True
        or ckpt.get(
            "feature_maps_detached"
        )
        is not True
    ):
        raise RuntimeError(
            f"STOP: seed {seed} adapter is not the decoupled frozen-base design."
        )

    ADAPTER_CHECKPOINTS[
        seed
    ] = {
        "path":
            path,
        "source":
            source,
        "checkpoint":
            ckpt,
    }

    print(
        f"✅ Adapter seed {seed}:",
        experiment_id,
        "|",
        source,
    )


✅ Adapter seed 42: M08_DecoupledLocalizationAdapter_Seed42 | direct
✅ Adapter seed 123: F02_LocalizationAdapter_Seed123 | direct
✅ Adapter seed 2026: F02_LocalizationAdapter_Seed2026 | direct


In [5]:
# ============================================================
# 6. Verify frozen calibration + selective policies
# ============================================================
_, m13_source, m13_scaler = (
    find_json_by_experiment_id(
        "temperature_scaler.json",
        CALIBRATION_EXPERIMENT_ID,
    )
)

if (
    m13_scaler
    is None
):
    raise FileNotFoundError(
        "Frozen M13 temperature_scaler.json not found."
    )

if (
    m13_scaler.get(
        "base_experiment_id"
    )
    != BASE_EXPERIMENT_IDS[
        42
    ]
):
    raise RuntimeError(
        "STOP: M13 is not tied to the seed-42 classifier."
    )

if (
    m13_scaler.get(
        "split_manifest_sha256"
    )
    != EXPECTED_SPLIT_MANIFEST_SHA256
):
    raise RuntimeError(
        "STOP: M13 split hash mismatch."
    )

if (
    m13_scaler.get(
        "official_test_used"
    )
    is not False
):
    raise RuntimeError(
        "STOP: M13 metadata does not confirm sealed test."
    )

TEMPERATURE = float(
    m13_scaler[
        "temperature"
    ]
)

if (
    not np.isfinite(
        TEMPERATURE
    )
    or TEMPERATURE
    <= 0
):
    raise RuntimeError(
        "STOP: frozen temperature is invalid."
    )


_, m14_source, m14_policy = (
    find_json_by_experiment_id(
        "selective_policy.json",
        NAIVE_SELECTIVE_EXPERIMENT_ID,
    )
)

if (
    m14_policy
    is None
):
    raise FileNotFoundError(
        "Frozen M14 selective_policy.json not found."
    )

_, m15_source, m15_policy = (
    find_json_by_experiment_id(
        "evidence_protected_selective_policy.json",
        PROTECTED_SELECTIVE_EXPERIMENT_ID,
    )
)

if (
    m15_policy
    is None
):
    raise FileNotFoundError(
        "Frozen M15 evidence-protected policy not found."
    )


for name, policy in [
    (
        "M14",
        m14_policy,
    ),
    (
        "M15",
        m15_policy,
    ),
]:
    if (
        policy.get(
            "base_experiment_id"
        )
        != BASE_EXPERIMENT_IDS[
            42
        ]
    ):
        raise RuntimeError(
            f"STOP: {name} is not tied to seed-42 classifier."
        )

    if (
        policy.get(
            "calibration_experiment_id"
        )
        != CALIBRATION_EXPERIMENT_ID
    ):
        raise RuntimeError(
            f"STOP: {name} is not tied to frozen M13 calibration."
        )

    if (
        not np.isclose(
            float(
                policy.get(
                    "temperature"
                )
            ),
            TEMPERATURE,
        )
    ):
        raise RuntimeError(
            f"STOP: {name} temperature mismatch."
        )

    if (
        policy.get(
            "official_test_used"
        )
        is not False
    ):
        raise RuntimeError(
            f"STOP: {name} metadata does not confirm sealed test."
        )


if (
    m15_policy.get(
        "ground_truth_used_to_define_policy"
    )
    is not False
    or m15_policy.get(
        "rarity_groups_used_to_define_policy"
    )
    is not False
):
    raise RuntimeError(
        "STOP: M15 protection policy definition is not label-independent."
    )

print(
    "✅ Frozen trustworthiness pipeline verified."
)
print(
    "Temperature:",
    f"{TEMPERATURE:.8f}",
)
print(
    "M13:",
    m13_source,
)
print(
    "M14:",
    m14_source,
)
print(
    "M15:",
    m15_source,
)


✅ Frozen trustworthiness pipeline verified.
Temperature: 0.41443756
M13: direct
M14: direct
M15: direct


In [6]:
# ============================================================
# 7. Verify frozen Head / Middle / Tail groups
# ============================================================
RARITY_PATH, RARITY_SOURCE = (
    locate_stage02_csv(
        "frozen_pathology_head_middle_tail_groups.csv"
    )
)

if (
    RARITY_PATH
    is None
):
    raise FileNotFoundError(
        "Frozen Stage-02 rarity-group table not found."
    )

rarity_groups = pd.read_csv(
    RARITY_PATH
)

LABEL_COLUMNS = list(
    REFERENCE_LABEL_COLUMNS
)

FINDING_CLASSES = list(
    REFERENCE_FINDING_CLASSES
)

PATHOLOGY_COLUMNS = [
    x
    for x in LABEL_COLUMNS
    if x
    != "No finding"
]

PATHOLOGY_INDICES = [
    LABEL_COLUMNS.index(
        x
    )
    for x in PATHOLOGY_COLUMNS
]

NO_FINDING_FINDING_INDEX = (
    FINDING_CLASSES.index(
        "No finding"
    )
)

if (
    set(
        rarity_groups[
            "label"
        ]
    )
    != set(
        PATHOLOGY_COLUMNS
    )
):
    raise RuntimeError(
        "STOP: Stage-02 rarity groups do not match frozen pathology labels."
    )

GROUP_TO_LABELS = {
    group:
        rarity_groups.loc[
            rarity_groups[
                "rarity_group"
            ]
            == group,
            "label",
        ].tolist()
    for group in [
        "Head",
        "Middle",
        "Tail",
    ]
}

GROUP_TO_INDICES = {
    group: [
        LABEL_COLUMNS.index(
            label
        )
        for label in labels
    ]
    for group, labels in GROUP_TO_LABELS.items()
}

print(
    "✅ Frozen rarity groups verified:",
    RARITY_SOURCE,
)
display(
    rarity_groups
)


✅ Frozen rarity groups verified: direct


,label,positive_images,frequency_rank,rarity_group
0,Bronchitis,842,1,Head
1,Brocho-pneumonia,545,2,Head
2,Bronchiolitis,497,3,Head
3,Other disease,412,4,Head
4,Pneumonia,392,5,Head
5,Hyaline membrane disease,19,6,Middle
6,Tuberculosis,14,7,Middle
7,Situs inversus,11,8,Middle
8,Mediastinal tumor,8,9,Middle
9,Pleuro-pneumonia,6,10,Middle


In [7]:
# ============================================================
# 8. Locate VinDr-PCXR root WITHOUT reading official test labels
# ============================================================
def find_vindr_root(
    search_root=KAGGLE_INPUT,
):
    candidates = []

    for f in search_root.rglob(
        "image_labels_train.csv"
    ):
        folder = (
            f.parent
        )

        expected = [
            "image_labels_train.csv",
            "image_labels_test.csv",
            "annotations_train.csv",
            "annotations_test.csv",
        ]

        names = {
            p.name
            for p in folder.iterdir()
            if p.is_file()
        }

        score = sum(
            x in names
            for x in expected
        )

        candidates.append(
            (
                score,
                folder,
            )
        )

    candidates.sort(
        key=lambda x: (
            -x[
                0
            ],
            len(
                str(
                    x[
                        1
                    ]
                )
            ),
        )
    )

    return candidates


vindr_candidates = (
    find_vindr_root()
)

if (
    not vindr_candidates
):
    raise FileNotFoundError(
        "VinDr-PCXR dataset root not found."
    )

DATA_ROOT = (
    vindr_candidates[
        0
    ][
        1
    ]
)

TEST_DICOM_DIR = (
    DATA_ROOT
    / "test"
)

for required_path in [
    DATA_ROOT
    / "image_labels_test.csv",
    DATA_ROOT
    / "annotations_test.csv",
    TEST_DICOM_DIR,
]:
    if (
        not required_path.exists()
    ):
        raise FileNotFoundError(
            "Required official-test resource missing: "
            + str(
                required_path
            )
        )

print(
    "VinDr-PCXR root located:",
    DATA_ROOT,
)
print(
    "Official-test resources exist, but labels have NOT been read yet."
)


VinDr-PCXR root located: /kaggle/input/datasets/kalloldaskushol/vindr-pcxr/Pediatric Chest X-ray/physionet.org/files/vindr-pcxr/1.0.0
Official-test resources exist, but labels have NOT been read yet.


In [8]:
# ============================================================
# 9. Write FINAL FREEZE DECLARATION before test-label access
# ============================================================
freeze_declaration = {
    "controller_id":
        CONTROLLER_ID,
    "official_test_status_before_authorization":
        "SEALED",
    "classifier_experiments":
        BASE_EXPERIMENT_IDS,
    "localization_adapter_experiments":
        ADAPTER_EXPERIMENT_IDS,
    "calibration_experiment":
        CALIBRATION_EXPERIMENT_ID,
    "temperature":
        TEMPERATURE,
    "naive_selective_experiment":
        NAIVE_SELECTIVE_EXPERIMENT_ID,
    "evidence_protected_selective_experiment":
        PROTECTED_SELECTIVE_EXPERIMENT_ID,
    "target_coverages_M14":
        m14_policy[
            "target_coverages"
        ],
    "target_coverages_M15":
        m15_policy[
            "target_coverages"
        ],
    "M15_protection_threshold":
        float(
            m15_policy[
                "protection_threshold"
            ]
        ),
    "disease_threshold_for_binary_metrics":
        FIXED_THRESHOLD,
    "disease_label_order":
        LABEL_COLUMNS,
    "finding_label_order":
        FINDING_CLASSES,
    "split_manifest_sha256":
        EXPECTED_SPLIT_MANIFEST_SHA256,
    "no_test_based_tuning_allowed":
        True,
    "no_test_based_recalibration_allowed":
        True,
    "no_test_based_policy_reselection_allowed":
        True,
    "no_posthoc_ensemble_introduced":
        True,
}

with open(
    OUTPUT_DIR
    / "FINAL_FREEZE_DECLARATION.json",
    "w",
) as f:
    json.dump(
        freeze_declaration,
        f,
        indent=2,
    )

print(
    "✅ FINAL DEVELOPMENT PIPELINE DECLARED FROZEN."
)
print(
    "Freeze declaration written before any official-test label read."
)


✅ FINAL DEVELOPMENT PIPELINE DECLARED FROZEN.
Freeze declaration written before any official-test label read.


# 10. FINAL AUTHORIZATION GATE

At this point:

- all model checkpoints are verified;
- all localization adapters are verified;
- M13 temperature is verified;
- M14 and M15 policies are verified;
- Head/Middle/Tail groups are verified;
- official-test files exist;
- **official-test labels still have not been read**.

## First run — sealed pre-test verification

Keep:

```python
AUTHORIZE_OFFICIAL_TEST = False
```

and use **Run All**.

The notebook will now finish **successfully** (no intentional exception), print:

> `PRE-TEST VERIFICATION COMPLETE`

and skip every official-test evaluation cell.

## Second run — irreversible final test

Only after the pre-test verification passes, change the single line to:

```python
AUTHORIZE_OFFICIAL_TEST = True
```

and use **Run All** again.

That second run opens the official test and performs the frozen final evaluation.

Afterward, no test-based tuning is permitted.


In [9]:
# ============================================================
# 11. CLEAN AUTHORIZATION GATE
# ============================================================
OFFICIAL_TEST_ACTIVE = (
    AUTHORIZE_OFFICIAL_TEST
    is True
)

if (
    OFFICIAL_TEST_ACTIVE
):
    print(
        "⚠️ OFFICIAL TEST AUTHORIZED."
    )
    print(
        "From the next cell onward, official test labels will be read."
    )
    print(
        "After this run: NO tuning, retraining, recalibration, "
        "threshold changes, checkpoint reselection, or seed selection."
    )

else:
    print(
        "✅ PRE-TEST VERIFICATION COMPLETE."
    )
    print(
        "Official test remains SEALED."
    )
    print(
        "All frozen artifacts and the final freeze declaration passed."
    )
    print(
        "No official-test labels have been read."
    )
    print(
        "When you intentionally want the one-time final evaluation, "
        "change AUTHORIZE_OFFICIAL_TEST = True in Cell 1 and Run All again."
    )


✅ PRE-TEST VERIFICATION COMPLETE.
Official test remains SEALED.
All frozen artifacts and the final freeze declaration passed.
No official-test labels have been read.
When you intentionally want the one-time final evaluation, change AUTHORIZE_OFFICIAL_TEST = True in Cell 1 and Run All again.


In [10]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 12. FIRST official-test label access
    # ============================================================
    test_labels = pd.read_csv(
        DATA_ROOT
        / "image_labels_test.csv"
    )

    test_annotations = pd.read_csv(
        DATA_ROOT
        / "annotations_test.csv"
    )

    if (
        len(
            test_labels
        )
        != EXPECTED_TEST_IMAGES
    ):
        raise RuntimeError(
            f"Expected {EXPECTED_TEST_IMAGES} official test rows, found {len(test_labels)}."
        )

    if (
        test_labels[
            "image_id"
        ].astype(
            str
        ).nunique()
        != EXPECTED_TEST_IMAGES
    ):
        raise RuntimeError(
            "Official test image_id is not one-to-one."
        )

    missing_disease_cols = (
        set(
            LABEL_COLUMNS
        )
        - set(
            test_labels.columns
        )
    )

    if (
        missing_disease_cols
    ):
        raise RuntimeError(
            "Official test disease labels do not match frozen label order: "
            + str(
                sorted(
                    missing_disease_cols
                )
            )
        )

    required_annotation_cols = {
        "image_id",
        "class_name",
        "x_min",
        "y_min",
        "x_max",
        "y_max",
    }

    if (
        required_annotation_cols
        - set(
            test_annotations.columns
        )
    ):
        raise RuntimeError(
            "Official test annotation columns are incomplete."
        )

    test_labels[
        "image_id"
    ] = (
        test_labels[
            "image_id"
        ].astype(
            str
        )
    )

    test_annotations[
        "image_id"
    ] = (
        test_annotations[
            "image_id"
        ].astype(
            str
        )
    )

    print(
        "Official test labels loaded:",
        len(
            test_labels
        ),
    )
    print(
        "Official test annotation rows:",
        len(
            test_annotations
        ),
    )
    print(
        "⚠️ TEST IS NOW OPENED. NO FURTHER TUNING IS PERMITTED."
    )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [11]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 13. Build frozen 37-finding test targets from class_name
    # ============================================================
    FINDING_TARGET_COLUMNS = [
        f"finding_target_{i:02d}"
        for i in range(
            len(
                FINDING_CLASSES
            )
        )
    ]

    unknown_test_findings = (
        set(
            test_annotations[
                "class_name"
            ]
            .dropna()
            .astype(
                str
            )
            .unique()
        )
        - set(
            FINDING_CLASSES
        )
    )

    if (
        unknown_test_findings
    ):
        raise RuntimeError(
            "Official test contains finding names absent from the frozen finding map: "
            + str(
                sorted(
                    unknown_test_findings
                )
            )
        )

    finding_matrix = pd.crosstab(
        test_annotations[
            "image_id"
        ],
        test_annotations[
            "class_name"
        ].astype(
            str
        ),
    )

    finding_matrix = (
        finding_matrix
        .reindex(
            columns=
                FINDING_CLASSES,
            fill_value=0,
        )
        .clip(
            upper=1
        )
        .astype(
            np.float32
        )
    )

    test_df = (
        test_labels.copy()
    )

    for target_col, class_name in zip(
        FINDING_TARGET_COLUMNS,
        FINDING_CLASSES,
    ):
        test_df[
            target_col
        ] = (
            test_df[
                "image_id"
            ]
            .map(
                finding_matrix[
                    class_name
                ]
            )
            .fillna(
                0
            )
            .astype(
                np.float32
            )
        )

    if (
        (
            test_df[
                FINDING_TARGET_COLUMNS
            ].sum(
                axis=1
            )
            <= 0
        )
        .any()
    ):
        raise RuntimeError(
            "At least one official-test image has no finding target."
        )

    print(
        "✅ Official-test finding targets built using class_name only."
    )
    print(
        "Numeric class_id used: NO"
    )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [12]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 14. Official-test DICOM index
    # ============================================================
    test_dicom_paths = {
        p.stem:
            p
        for p in TEST_DICOM_DIR.rglob(
            "*"
        )
        if (
            p.is_file()
            and p.suffix.lower()
            in {
                ".dcm",
                ".dicom",
            }
        )
    }

    test_ids = set(
        test_df[
            "image_id"
        ].tolist()
    )

    missing_test_dicoms = (
        test_ids
        - set(
            test_dicom_paths.keys()
        )
    )

    if (
        missing_test_dicoms
    ):
        raise FileNotFoundError(
            "Some official-test DICOMs are missing."
        )

    print(
        "Official-test DICOM files indexed:",
        len(
            test_dicom_paths
        ),
    )
    print(
        "Required test IDs:",
        len(
            test_ids
        ),
    )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [13]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 15. Prepare official-test normalized valid boxes
    # ============================================================
    BOX_COLS = [
        "x_min",
        "y_min",
        "x_max",
        "y_max",
    ]

    box_numeric = (
        test_annotations[
            BOX_COLS
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    valid_box_flag = (
        box_numeric.notna()
        .all(
            axis=1
        )
        & (
            box_numeric[
                "x_max"
            ]
            > box_numeric[
                "x_min"
            ]
        )
        & (
            box_numeric[
                "y_max"
            ]
            > box_numeric[
                "y_min"
            ]
        )
        & (
            box_numeric[
                BOX_COLS
            ]
            >= 0
        ).all(
            axis=1
        )
    )

    valid_boxes = (
        test_annotations.loc[
            valid_box_flag
        ]
        .copy()
    )

    for c in BOX_COLS:
        valid_boxes[
            c
        ] = (
            box_numeric.loc[
                valid_box_flag,
                c,
            ].astype(
                float
            )
        )

    valid_boxes = (
        valid_boxes[
            valid_boxes[
                "class_name"
            ].astype(
                str
            )
            != "No finding"
        ]
        .copy()
    )

    valid_boxes = (
        valid_boxes[
            valid_boxes[
                "image_id"
            ].isin(
                test_ids
            )
        ]
        .copy()
    )

    boxed_ids = sorted(
        valid_boxes[
            "image_id"
        ]
        .unique()
        .tolist()
    )

    dimension_rows = []

    for image_id in tqdm(
        boxed_ids,
        desc=
            "Reading official-test boxed-image dimensions",
    ):
        ds = pydicom.dcmread(
            str(
                test_dicom_paths[
                    image_id
                ]
            ),
            stop_before_pixels=True,
            specific_tags=[
                "Rows",
                "Columns",
            ],
        )

        dimension_rows.append({
            "image_id":
                image_id,
            "rows":
                int(
                    ds.Rows
                ),
            "columns":
                int(
                    ds.Columns
                ),
        })

    dimensions_df = pd.DataFrame(
        dimension_rows
    )

    valid_boxes = valid_boxes.merge(
        dimensions_df,
        on="image_id",
        how="left",
        validate="many_to_one",
    )

    valid_boxes[
        "nx_min"
    ] = (
        valid_boxes[
            "x_min"
        ]
        / valid_boxes[
            "columns"
        ]
    ).clip(
        0,
        1,
    )

    valid_boxes[
        "nx_max"
    ] = (
        valid_boxes[
            "x_max"
        ]
        / valid_boxes[
            "columns"
        ]
    ).clip(
        0,
        1,
    )

    valid_boxes[
        "ny_min"
    ] = (
        valid_boxes[
            "y_min"
        ]
        / valid_boxes[
            "rows"
        ]
    ).clip(
        0,
        1,
    )

    valid_boxes[
        "ny_max"
    ] = (
        valid_boxes[
            "y_max"
        ]
        / valid_boxes[
            "rows"
        ]
    ).clip(
        0,
        1,
    )

    if (
        not (
            (
                valid_boxes[
                    "nx_max"
                ]
                > valid_boxes[
                    "nx_min"
                ]
            )
            & (
                valid_boxes[
                    "ny_max"
                ]
                > valid_boxes[
                    "ny_min"
                ]
            )
        ).all()
    ):
        raise RuntimeError(
            "A test box became invalid after normalization."
        )

    finding_to_index = {
        name:
            i
        for i, name in enumerate(
            FINDING_CLASSES
        )
    }

    BOXES_BY_IMAGE = {}

    for image_id, group in valid_boxes.groupby(
        "image_id"
    ):
        BOXES_BY_IMAGE[
            image_id
        ] = [
            (
                finding_to_index[
                    str(
                        row.class_name
                    )
                ],
                float(
                    row.nx_min
                ),
                float(
                    row.ny_min
                ),
                float(
                    row.nx_max
                ),
                float(
                    row.ny_max
                ),
            )
            for row in group.itertuples(
                index=False
            )
        ]

    print(
        "Valid official-test box rows:",
        len(
            valid_boxes
        ),
    )
    print(
        "Official-test images with valid abnormal boxes:",
        len(
            BOXES_BY_IMAGE
        ),
    )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [14]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 16. Official-test DICOM preprocessing cache
    # ============================================================
    def dicom_to_uint8_320(
        path,
        output_size=IMAGE_SIZE,
    ):
        ds = pydicom.dcmread(
            str(
                path
            )
        )

        arr = ds.pixel_array

        try:
            arr = apply_modality_lut(
                arr,
                ds,
            )
        except Exception:
            arr = np.asarray(
                arr
            )

        try:
            arr = apply_voi_lut(
                arr,
                ds,
            )
        except Exception:
            arr = np.asarray(
                arr
            )

        arr = np.asarray(
            arr,
            dtype=np.float32,
        )

        if (
            not np.isfinite(
                arr
            ).all()
        ):
            finite = arr[
                np.isfinite(
                    arr
                )
            ]

            fill = (
                float(
                    np.median(
                        finite
                    )
                )
                if finite.size
                else 0.0
            )

            arr = np.nan_to_num(
                arr,
                nan=fill,
                posinf=fill,
                neginf=fill,
            )

        if (
            str(
                getattr(
                    ds,
                    "PhotometricInterpretation",
                    "",
                )
            ).upper()
            == "MONOCHROME1"
        ):
            arr = (
                arr.max()
                + arr.min()
                - arr
            )

        lo, hi = np.percentile(
            arr,
            [
                0.5,
                99.5,
            ],
        )

        if (
            not np.isfinite(
                lo
            )
            or not np.isfinite(
                hi
            )
            or hi
            <= lo
        ):
            lo = float(
                arr.min()
            )
            hi = float(
                arr.max()
            )

        if (
            hi
            > lo
        ):
            arr = np.clip(
                arr,
                lo,
                hi,
            )

            arr = (
                arr
                - lo
            ) / (
                hi
                - lo
            )
        else:
            arr = np.zeros_like(
                arr,
                dtype=np.float32,
            )

        arr = (
            arr
            * 255.0
        ).round().astype(
            np.uint8
        )

        image = Image.fromarray(
            arr
        )

        image = image.resize(
            (
                output_size,
                output_size,
            ),
            resample=
                Image.Resampling.BILINEAR,
        )

        return np.asarray(
            image,
            dtype=np.uint8,
        )


    preprocess_failures = []

    for image_id in tqdm(
        test_df[
            "image_id"
        ].tolist(),
        desc=
            "Caching official-test X-rays",
    ):
        target = (
            CACHE_DIR
            / f"{image_id}.npy"
        )

        if (
            target.exists()
        ):
            continue

        try:
            arr = dicom_to_uint8_320(
                test_dicom_paths[
                    image_id
                ]
            )

            np.save(
                target,
                arr,
                allow_pickle=False,
            )

        except Exception as e:
            preprocess_failures.append({
                "error":
                    f"{type(e).__name__}: {e}",
            })

    if (
        preprocess_failures
    ):
        raise RuntimeError(
            "Official-test preprocessing failed for one or more images."
        )

    print(
        "✅ Official-test image cache complete."
    )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [15]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 17. Official-test dataset — NO augmentation
    # ============================================================
    IMAGENET_MEAN = [
        0.485,
        0.456,
        0.406,
    ]

    # Intentionally preserved from executed Stage-09/10 pipeline.
    IMAGENET_STD = [
        0.229,
        0.320,
        0.225,
    ]


    def rasterize_box_masks(
        image_id,
    ):
        masks = torch.zeros(
            (
                len(
                    FINDING_CLASSES
                ),
                BOX_MASK_SIZE,
                BOX_MASK_SIZE,
            ),
            dtype=torch.uint8,
        )

        for (
            class_index,
            nx_min,
            ny_min,
            nx_max,
            ny_max,
        ) in BOXES_BY_IMAGE.get(
            str(
                image_id
            ),
            [],
        ):
            x1 = int(
                np.floor(
                    nx_min
                    * BOX_MASK_SIZE
                )
            )
            y1 = int(
                np.floor(
                    ny_min
                    * BOX_MASK_SIZE
                )
            )
            x2 = int(
                np.ceil(
                    nx_max
                    * BOX_MASK_SIZE
                )
            )
            y2 = int(
                np.ceil(
                    ny_max
                    * BOX_MASK_SIZE
                )
            )

            x1 = int(
                np.clip(
                    x1,
                    0,
                    BOX_MASK_SIZE
                    - 1,
                )
            )
            y1 = int(
                np.clip(
                    y1,
                    0,
                    BOX_MASK_SIZE
                    - 1,
                )
            )
            x2 = int(
                np.clip(
                    x2,
                    x1
                    + 1,
                    BOX_MASK_SIZE,
                )
            )
            y2 = int(
                np.clip(
                    y2,
                    y1
                    + 1,
                    BOX_MASK_SIZE,
                )
            )

            masks[
                class_index,
                y1:y2,
                x1:x2,
            ] = 1

        return masks


    class OfficialTestDataset(
        Dataset
    ):
        def __init__(
            self,
            dataframe,
        ):
            self.df = (
                dataframe
                .reset_index(
                    drop=True
                )
            )

        def __len__(
            self,
        ):
            return len(
                self.df
            )

        def __getitem__(
            self,
            idx,
        ):
            row = self.df.iloc[
                idx
            ]

            image_id = str(
                row[
                    "image_id"
                ]
            )

            arr = np.load(
                CACHE_DIR
                / f"{image_id}.npy",
                allow_pickle=False,
            )

            image = (
                torch.from_numpy(
                    arr
                )
                .float()
                .unsqueeze(
                    0
                )
                / 255.0
            )

            image = image.repeat(
                3,
                1,
                1,
            )

            image = TF.normalize(
                image,
                mean=
                    IMAGENET_MEAN,
                std=
                    IMAGENET_STD,
            )

            disease_target = torch.tensor(
                row[
                    LABEL_COLUMNS
                ].to_numpy(
                    dtype=np.float32
                ),
                dtype=torch.float32,
            )

            finding_target = torch.tensor(
                row[
                    FINDING_TARGET_COLUMNS
                ].to_numpy(
                    dtype=np.float32
                ),
                dtype=torch.float32,
            )

            box_masks = (
                rasterize_box_masks(
                    image_id
                )
            )

            box_present = (
                box_masks
                .flatten(
                    1
                )
                .any(
                    dim=1
                )
            )

            box_present[
                NO_FINDING_FINDING_INDEX
            ] = False

            return (
                image,
                disease_target,
                finding_target,
                box_masks,
                box_present,
            )


    test_dataset = (
        OfficialTestDataset(
            test_df
        )
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=
            BATCH_SIZE,
        shuffle=False,
        num_workers=
            NUM_WORKERS,
        pin_memory=True,
        persistent_workers=
            (
                NUM_WORKERS
                > 0
            ),
    )

    sample = next(
        iter(
            test_loader
        )
    )

    if (
        sample[
            0
        ].shape[
            1:
        ]
        != (
            3,
            IMAGE_SIZE,
            IMAGE_SIZE,
        )
    ):
        raise RuntimeError(
            "Official-test tensor shape is incorrect."
        )

    print(
        "✅ Official-test DataLoader ready."
    )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [16]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 18. Frozen base + adapter architecture
    # ============================================================
    class FrozenMultitaskBase(
        nn.Module
    ):
        def __init__(
            self,
            n_disease_labels,
            n_finding_labels,
        ):
            super().__init__()

            self.backbone = resnet50(
                weights=None
            )

            feature_dim = (
                self.backbone
                .fc
                .in_features
            )

            self.backbone.fc = (
                nn.Identity()
            )

            self.disease_head = nn.Linear(
                feature_dim,
                n_disease_labels,
            )

            self.finding_head = nn.Linear(
                feature_dim,
                n_finding_labels,
            )

        def extract_spatial_features(
            self,
            x,
        ):
            b = self.backbone

            x = b.conv1(
                x
            )
            x = b.bn1(
                x
            )
            x = b.relu(
                x
            )
            x = b.maxpool(
                x
            )
            x = b.layer1(
                x
            )
            x = b.layer2(
                x
            )
            x = b.layer3(
                x
            )
            x = b.layer4(
                x
            )

            return x

        def forward(
            self,
            x,
        ):
            feature_maps = (
                self.extract_spatial_features(
                    x
                )
            )

            pooled = (
                self.backbone.avgpool(
                    feature_maps
                )
            )

            pooled = torch.flatten(
                pooled,
                1,
            )

            disease_logits = (
                self.disease_head(
                    pooled
                )
            )

            finding_logits = (
                self.finding_head(
                    pooled
                )
            )

            return (
                disease_logits,
                finding_logits,
                feature_maps,
            )


    class LocalizationAdapter(
        nn.Module
    ):
        def __init__(
            self,
            in_channels=2048,
            hidden_channels=256,
            n_findings=37,
        ):
            super().__init__()

            self.reduce = nn.Conv2d(
                in_channels,
                hidden_channels,
                kernel_size=1,
                bias=False,
            )

            self.norm = nn.GroupNorm(
                num_groups=32,
                num_channels=
                    hidden_channels,
            )

            self.activation = nn.ReLU(
                inplace=True
            )

            self.class_maps = nn.Conv2d(
                hidden_channels,
                n_findings,
                kernel_size=1,
                bias=True,
            )

        def forward(
            self,
            detached_feature_maps,
        ):
            x = self.reduce(
                detached_feature_maps
            )

            x = self.norm(
                x
            )

            x = self.activation(
                x
            )

            class_maps = (
                self.class_maps(
                    x
                )
            )

            local_finding_logits = (
                F.adaptive_avg_pool2d(
                    class_maps,
                    output_size=1,
                )
                .flatten(
                    1
                )
            )

            return (
                local_finding_logits,
                class_maps,
            )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [17]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 19. Metric + localization helpers
    # ============================================================
    def safe_ap(
        y_true,
        y_prob,
    ):
        if (
            y_true.sum()
            == 0
        ):
            return np.nan

        try:
            return float(
                average_precision_score(
                    y_true,
                    y_prob,
                )
            )
        except Exception:
            return np.nan


    def safe_auc(
        y_true,
        y_prob,
    ):
        if (
            np.unique(
                y_true
            ).size
            < 2
        ):
            return np.nan

        try:
            return float(
                roc_auc_score(
                    y_true,
                    y_prob,
                )
            )
        except Exception:
            return np.nan


    def binary_specificity(
        y_true,
        y_pred,
    ):
        cm = confusion_matrix(
            y_true,
            y_pred,
            labels=[
                0,
                1,
            ],
        )

        tn, fp, fn, tp = (
            cm.ravel()
        )

        return (
            float(
                tn
                / (
                    tn
                    + fp
                )
            )
            if (
                tn
                + fp
            )
            > 0
            else np.nan
        )


    def disease_metrics(
        y_true,
        y_prob,
    ):
        y_pred = (
            y_prob
            >= FIXED_THRESHOLD
        ).astype(
            int
        )

        rows = []

        for i, label in enumerate(
            LABEL_COLUMNS
        ):
            yt = (
                y_true[
                    :,
                    i,
                ].astype(
                    int
                )
            )

            yp = (
                y_prob[
                    :,
                    i
                ]
            )

            yh = (
                y_pred[
                    :,
                    i
                ]
            )

            rows.append({
                "label":
                    label,
                "positive_test_images":
                    int(
                        yt.sum()
                    ),
                "AUPRC":
                    safe_ap(
                        yt,
                        yp,
                    ),
                "AUROC":
                    safe_auc(
                        yt,
                        yp,
                    ),
                "F1_at_0.5":
                    float(
                        f1_score(
                            yt,
                            yh,
                            zero_division=0,
                        )
                    ),
                "Sensitivity_at_0.5":
                    float(
                        recall_score(
                            yt,
                            yh,
                            zero_division=0,
                        )
                    ),
                "Specificity_at_0.5":
                    binary_specificity(
                        yt,
                        yh,
                    ),
            })

        table = pd.DataFrame(
            rows
        )

        pathology = (
            table[
                table[
                    "label"
                ].isin(
                    PATHOLOGY_COLUMNS
                )
            ]
        )

        summary = {
            "pathology14_macro_AUPRC":
                float(
                    pathology[
                        "AUPRC"
                    ].mean(
                        skipna=True
                    )
                ),
            "pathology14_macro_AUROC":
                float(
                    pathology[
                        "AUROC"
                    ].mean(
                        skipna=True
                    )
                ),
            "pathology14_macro_F1_at_0.5":
                float(
                    pathology[
                        "F1_at_0.5"
                    ].mean(
                        skipna=True
                    )
                ),
            "pathology14_macro_sensitivity_at_0.5":
                float(
                    pathology[
                        "Sensitivity_at_0.5"
                    ].mean(
                        skipna=True
                    )
                ),
        }

        group_rows = []

        for group in [
            "Head",
            "Middle",
            "Tail",
        ]:
            subset = (
                table[
                    table[
                        "label"
                    ].isin(
                        GROUP_TO_LABELS[
                            group
                        ]
                    )
                ]
            )

            group_rows.append({
                "group":
                    group,
                "macro_AUPRC":
                    float(
                        subset[
                            "AUPRC"
                        ].mean(
                            skipna=True
                        )
                    ),
                "macro_AUROC":
                    float(
                        subset[
                            "AUROC"
                        ].mean(
                            skipna=True
                        )
                    ),
            })

        return (
            table,
            summary,
            pd.DataFrame(
                group_rows
            ),
        )


    def finding_metrics(
        y_true,
        y_prob,
    ):
        rows = []

        for i, class_name in enumerate(
            FINDING_CLASSES
        ):
            yt = (
                y_true[
                    :,
                    i,
                ].astype(
                    int
                )
            )

            yp = (
                y_prob[
                    :,
                    i
                ]
            )

            rows.append({
                "finding_index":
                    i,
                "class_name":
                    class_name,
                "positive_test_images":
                    int(
                        yt.sum()
                    ),
                "AUPRC":
                    safe_ap(
                        yt,
                        yp,
                    ),
                "AUROC":
                    safe_auc(
                        yt,
                        yp,
                    ),
            })

        table = pd.DataFrame(
            rows
        )

        actual = (
            table[
                table[
                    "class_name"
                ]
                != "No finding"
            ]
        )

        return (
            table,
            {
                "actual36_macro_AUPRC":
                    float(
                        actual[
                            "AUPRC"
                        ].mean(
                            skipna=True
                        )
                    ),
                "actual36_macro_AUROC":
                    float(
                        actual[
                            "AUROC"
                        ].mean(
                            skipna=True
                        )
                    ),
            },
        )


    def maps_to_attention(
        class_maps,
    ):
        b, k, h, w = (
            class_maps.shape
        )

        return torch.softmax(
            class_maps.flatten(
                2
            ),
            dim=-1,
        ).view(
            b,
            k,
            h,
            w,
        )


    def localization_batch_stats(
        class_maps,
        finding_targets,
        box_masks,
        box_present,
    ):
        attention = maps_to_attention(
            class_maps
        )

        h, w = (
            attention.shape[
                -2:
            ]
        )

        target_masks = (
            F.adaptive_max_pool2d(
                box_masks.float(),
                output_size=(
                    h,
                    w,
                ),
            )
            > 0
        ).float()

        valid = (
            box_present.bool()
            & (
                finding_targets
                > 0.5
            )
        )

        valid[
            :,
            NO_FINDING_FINDING_INDEX,
        ] = False

        attention_mass = (
            attention
            * target_masks
        ).sum(
            dim=(
                -2,
                -1,
            )
        )

        max_indices = (
            attention.flatten(
                2
            )
            .argmax(
                dim=-1,
                keepdim=True,
            )
        )

        pointing_hit = (
            target_masks.flatten(
                2
            )
            .gather(
                dim=2,
                index=
                    max_indices,
            )
            .squeeze(
                -1
            )
        )

        return (
            valid,
            attention_mass,
            pointing_hit,
        )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [18]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 20. Evaluate one seed on official test
    # ============================================================
    @torch.no_grad()
    def evaluate_seed(
        seed,
    ):
        base_ckpt = (
            BASE_CHECKPOINTS[
                seed
            ][
                "checkpoint"
            ]
        )

        adapter_ckpt = (
            ADAPTER_CHECKPOINTS[
                seed
            ][
                "checkpoint"
            ]
        )

        base_model = (
            FrozenMultitaskBase(
                n_disease_labels=
                    len(
                        LABEL_COLUMNS
                    ),
                n_finding_labels=
                    len(
                        FINDING_CLASSES
                    ),
            )
            .to(
                DEVICE
            )
        )

        base_model.load_state_dict(
            base_ckpt[
                "model_state_dict"
            ],
            strict=True,
        )

        for p in base_model.parameters():
            p.requires_grad = False

        base_model.eval()

        adapter = (
            LocalizationAdapter(
                in_channels=2048,
                hidden_channels=256,
                n_findings=
                    len(
                        FINDING_CLASSES
                    ),
            )
            .to(
                DEVICE
            )
        )

        adapter.load_state_dict(
            adapter_ckpt[
                "localization_adapter_state_dict"
            ],
            strict=True,
        )

        for p in adapter.parameters():
            p.requires_grad = False

        adapter.eval()

        disease_true = []
        disease_logits_all = []
        disease_prob = []

        finding_true = []
        base_finding_prob = []
        adapter_finding_prob = []

        n_findings = len(
            FINDING_CLASSES
        )

        direct_count = np.zeros(
            n_findings,
            dtype=np.int64,
        )

        direct_mass_sum = np.zeros(
            n_findings,
            dtype=np.float64,
        )

        direct_point_sum = np.zeros(
            n_findings,
            dtype=np.float64,
        )

        adapter_count = np.zeros(
            n_findings,
            dtype=np.int64,
        )

        adapter_mass_sum = np.zeros(
            n_findings,
            dtype=np.float64,
        )

        adapter_point_sum = np.zeros(
            n_findings,
            dtype=np.float64,
        )

        for (
            images,
            disease_targets,
            finding_targets,
            box_masks,
            box_present,
        ) in tqdm(
            test_loader,
            desc=
                f"Official test seed {seed}",
            leave=False,
        ):
            images = images.to(
                DEVICE,
                non_blocking=True,
            )

            (
                disease_logits,
                finding_logits,
                feature_maps,
            ) = base_model(
                images
            )

            (
                local_logits,
                adapter_maps,
            ) = adapter(
                feature_maps.detach()
            )

            direct_maps = torch.einsum(
                "kc,bchw->bkhw",
                base_model.finding_head.weight,
                feature_maps,
            )

            finding_targets_gpu = (
                finding_targets.to(
                    DEVICE
                )
            )

            box_masks_gpu = (
                box_masks.to(
                    DEVICE
                )
            )

            box_present_gpu = (
                box_present.to(
                    DEVICE
                )
            )

            (
                direct_valid,
                direct_mass,
                direct_point,
            ) = localization_batch_stats(
                direct_maps,
                finding_targets_gpu,
                box_masks_gpu,
                box_present_gpu,
            )

            (
                adapter_valid,
                adapter_mass,
                adapter_point,
            ) = localization_batch_stats(
                adapter_maps,
                finding_targets_gpu,
                box_masks_gpu,
                box_present_gpu,
            )

            for (
                valid,
                mass,
                point,
                count_acc,
                mass_acc,
                point_acc,
            ) in [
                (
                    direct_valid,
                    direct_mass,
                    direct_point,
                    direct_count,
                    direct_mass_sum,
                    direct_point_sum,
                ),
                (
                    adapter_valid,
                    adapter_mass,
                    adapter_point,
                    adapter_count,
                    adapter_mass_sum,
                    adapter_point_sum,
                ),
            ]:
                valid_np = (
                    valid
                    .cpu()
                    .numpy()
                )

                mass_np = (
                    mass
                    .float()
                    .cpu()
                    .numpy()
                )

                point_np = (
                    point
                    .float()
                    .cpu()
                    .numpy()
                )

                count_acc += (
                    valid_np.sum(
                        axis=0
                    )
                )

                mass_acc += (
                    mass_np
                    * valid_np
                ).sum(
                    axis=0
                )

                point_acc += (
                    point_np
                    * valid_np
                ).sum(
                    axis=0
                )

            disease_true.append(
                disease_targets.numpy()
            )

            disease_logits_all.append(
                disease_logits
                .float()
                .cpu()
                .numpy()
            )

            disease_prob.append(
                torch.sigmoid(
                    disease_logits
                )
                .float()
                .cpu()
                .numpy()
            )

            finding_true.append(
                finding_targets.numpy()
            )

            base_finding_prob.append(
                torch.sigmoid(
                    finding_logits
                )
                .float()
                .cpu()
                .numpy()
            )

            adapter_finding_prob.append(
                torch.sigmoid(
                    local_logits
                )
                .float()
                .cpu()
                .numpy()
            )

        disease_true = np.concatenate(
            disease_true,
            axis=0,
        )

        disease_logits_all = np.concatenate(
            disease_logits_all,
            axis=0,
        )

        disease_prob = np.concatenate(
            disease_prob,
            axis=0,
        )

        finding_true = np.concatenate(
            finding_true,
            axis=0,
        )

        base_finding_prob = np.concatenate(
            base_finding_prob,
            axis=0,
        )

        adapter_finding_prob = np.concatenate(
            adapter_finding_prob,
            axis=0,
        )

        (
            disease_per_class,
            disease_summary,
            disease_groups,
        ) = disease_metrics(
            disease_true,
            disease_prob,
        )

        (
            base_finding_per_class,
            base_finding_summary,
        ) = finding_metrics(
            finding_true,
            base_finding_prob,
        )

        (
            adapter_finding_per_class,
            adapter_finding_summary,
        ) = finding_metrics(
            finding_true,
            adapter_finding_prob,
        )

        localization_rows = []

        for i, class_name in enumerate(
            FINDING_CLASSES
        ):
            n = int(
                adapter_count[
                    i
                ]
            )

            localization_rows.append({
                "finding_index":
                    i,
                "class_name":
                    class_name,
                "boxed_test_pairs":
                    n,
                "direct_attention_mass":
                    (
                        float(
                            direct_mass_sum[
                                i
                            ]
                            / direct_count[
                                i
                            ]
                        )
                        if direct_count[
                            i
                        ]
                        else np.nan
                    ),
                "direct_pointing_accuracy":
                    (
                        float(
                            direct_point_sum[
                                i
                            ]
                            / direct_count[
                                i
                            ]
                        )
                        if direct_count[
                            i
                        ]
                        else np.nan
                    ),
                "adapter_attention_mass":
                    (
                        float(
                            adapter_mass_sum[
                                i
                            ]
                            / n
                        )
                        if n
                        else np.nan
                    ),
                "adapter_pointing_accuracy":
                    (
                        float(
                            adapter_point_sum[
                                i
                            ]
                            / n
                        )
                        if n
                        else np.nan
                    ),
            })

        localization_per_finding = (
            pd.DataFrame(
                localization_rows
            )
        )

        valid_loc = (
            localization_per_finding[
                localization_per_finding[
                    "boxed_test_pairs"
                ]
                > 0
            ]
        )

        total_pairs = int(
            valid_loc[
                "boxed_test_pairs"
            ].sum()
        )

        def weighted_mean(
            col,
        ):
            return float(
                (
                    valid_loc[
                        col
                    ]
                    * valid_loc[
                        "boxed_test_pairs"
                    ]
                ).sum()
                / total_pairs
            )

        localization_summary = {
            "boxed_test_pairs":
                total_pairs,
            "findings_with_test_boxes":
                int(
                    len(
                        valid_loc
                    )
                ),
            "direct_attention_mass":
                weighted_mean(
                    "direct_attention_mass"
                ),
            "direct_pointing_accuracy":
                weighted_mean(
                    "direct_pointing_accuracy"
                ),
            "adapter_attention_mass":
                weighted_mean(
                    "adapter_attention_mass"
                ),
            "adapter_pointing_accuracy":
                weighted_mean(
                    "adapter_pointing_accuracy"
                ),
        }

        localization_summary[
            "attention_mass_gain"
        ] = (
            localization_summary[
                "adapter_attention_mass"
            ]
            - localization_summary[
                "direct_attention_mass"
            ]
        )

        localization_summary[
            "pointing_gain"
        ] = (
            localization_summary[
                "adapter_pointing_accuracy"
            ]
            - localization_summary[
                "direct_pointing_accuracy"
            ]
        )

        result = {
            "seed":
                seed,
            "base_experiment_id":
                BASE_EXPERIMENT_IDS[
                    seed
                ],
            "adapter_experiment_id":
                ADAPTER_EXPERIMENT_IDS[
                    seed
                ],
            **disease_summary,
            **{
                "head_macro_AUPRC":
                    float(
                        disease_groups.loc[
                            disease_groups[
                                "group"
                            ]
                            == "Head",
                            "macro_AUPRC",
                        ].iloc[
                            0
                        ]
                    ),
                "middle_macro_AUPRC":
                    float(
                        disease_groups.loc[
                            disease_groups[
                                "group"
                            ]
                            == "Middle",
                            "macro_AUPRC",
                        ].iloc[
                            0
                        ]
                    ),
                "tail_macro_AUPRC":
                    float(
                        disease_groups.loc[
                            disease_groups[
                                "group"
                            ]
                            == "Tail",
                            "macro_AUPRC",
                        ].iloc[
                            0
                        ]
                    ),
                "base_finding36_macro_AUPRC":
                    base_finding_summary[
                        "actual36_macro_AUPRC"
                    ],
                "adapter_finding36_macro_AUPRC":
                    adapter_finding_summary[
                        "actual36_macro_AUPRC"
                    ],
                **localization_summary,
            },
        }

        del (
            base_model,
            adapter,
        )

        gc.collect()
        torch.cuda.empty_cache()

        return {
            "result":
                result,
            "disease_per_class":
                disease_per_class,
            "disease_groups":
                disease_groups,
            "localization_per_finding":
                localization_per_finding,
            "adapter_finding_per_class":
                adapter_finding_per_class,
            "disease_true":
                disease_true,
            "disease_logits":
                disease_logits_all,
            "disease_prob":
                disease_prob,
        }
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [19]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 21. Run frozen official-test evaluation for all 3 seeds
    # ============================================================
    seed_outputs = {}

    for seed in SEEDS:
        print(
            "\n"
            + "="
            * 86
        )
        print(
            "OFFICIAL TEST — FROZEN SEED",
            seed,
        )
        print(
            "="
            * 86
        )

        seed_outputs[
            seed
        ] = evaluate_seed(
            seed
        )

    print(
        "✅ All three frozen classifier/localization seeds evaluated."
    )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [20]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 22. Classification 3-seed official-test summary
    # ============================================================
    classification_three_seed = pd.DataFrame(
        [
            seed_outputs[
                seed
            ][
                "result"
            ]
            for seed in SEEDS
        ]
    )

    classification_three_seed.to_csv(
        TABLE_DIR
        / "official_test_three_seed_results.csv",
        index=False,
    )

    classification_metrics = [
        "pathology14_macro_AUPRC",
        "pathology14_macro_AUROC",
        "pathology14_macro_F1_at_0.5",
        "pathology14_macro_sensitivity_at_0.5",
        "head_macro_AUPRC",
        "middle_macro_AUPRC",
        "tail_macro_AUPRC",
    ]

    summary_rows = []

    for metric in classification_metrics:
        values = (
            classification_three_seed[
                metric
            ].astype(
                float
            )
        )

        summary_rows.append({
            "metric":
                metric,
            "mean":
                float(
                    values.mean()
                ),
            "std_sample":
                float(
                    values.std(
                        ddof=1
                    )
                ),
            "min":
                float(
                    values.min()
                ),
            "max":
                float(
                    values.max()
                ),
            "n_seeds":
                3,
        })

    classification_mean_sd = pd.DataFrame(
        summary_rows
    )

    classification_mean_sd.to_csv(
        TABLE_DIR
        / "official_test_classification_three_seed_mean_sd.csv",
        index=False,
    )

    display(
        classification_three_seed[
            [
                "seed",
                "pathology14_macro_AUPRC",
                "pathology14_macro_AUROC",
                "pathology14_macro_F1_at_0.5",
                "head_macro_AUPRC",
                "middle_macro_AUPRC",
                "tail_macro_AUPRC",
            ]
        ]
    )

    display(
        classification_mean_sd
    )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [21]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 23. Per-class 3-seed official-test summary
    # ============================================================
    per_class_long = []

    for seed in SEEDS:
        temp = (
            seed_outputs[
                seed
            ][
                "disease_per_class"
            ]
            .copy()
        )

        temp[
            "seed"
        ] = (
            seed
        )

        per_class_long.append(
            temp
        )

    per_class_long = pd.concat(
        per_class_long,
        ignore_index=True,
    )

    # Aggregate only; the table contains class-level metrics, not patient rows.
    per_class_mean_sd = (
        per_class_long
        .groupby(
            [
                "label",
                "positive_test_images",
            ],
            as_index=False,
        )
        .agg(
            AUPRC_mean=(
                "AUPRC",
                "mean",
            ),
            AUPRC_std=(
                "AUPRC",
                lambda x:
                    x.std(
                        ddof=1
                    ),
            ),
            AUROC_mean=(
                "AUROC",
                "mean",
            ),
            AUROC_std=(
                "AUROC",
                lambda x:
                    x.std(
                        ddof=1
                    ),
            ),
            F1_mean=(
                "F1_at_0.5",
                "mean",
            ),
            F1_std=(
                "F1_at_0.5",
                lambda x:
                    x.std(
                        ddof=1
                    ),
            ),
            Sensitivity_mean=(
                "Sensitivity_at_0.5",
                "mean",
            ),
            Sensitivity_std=(
                "Sensitivity_at_0.5",
                lambda x:
                    x.std(
                        ddof=1
                    ),
            ),
        )
    )

    per_class_mean_sd.to_csv(
        TABLE_DIR
        / "official_test_per_class_three_seed_mean_sd.csv",
        index=False,
    )

    display(
        per_class_mean_sd
    )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [22]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 24. Localization 3-seed official-test summary
    # ============================================================
    localization_columns = [
        "seed",
        "direct_attention_mass",
        "adapter_attention_mass",
        "attention_mass_gain",
        "direct_pointing_accuracy",
        "adapter_pointing_accuracy",
        "pointing_gain",
        "adapter_finding36_macro_AUPRC",
        "boxed_test_pairs",
    ]

    localization_three_seed = (
        classification_three_seed[
            localization_columns
        ].copy()
    )

    localization_three_seed.to_csv(
        TABLE_DIR
        / "official_test_localization_three_seed_results.csv",
        index=False,
    )

    loc_metric_cols = [
        "direct_attention_mass",
        "adapter_attention_mass",
        "attention_mass_gain",
        "direct_pointing_accuracy",
        "adapter_pointing_accuracy",
        "pointing_gain",
        "adapter_finding36_macro_AUPRC",
    ]

    loc_summary_rows = []

    for metric in loc_metric_cols:
        values = pd.to_numeric(
            localization_three_seed[
                metric
            ],
            errors="coerce",
        )

        finite = (
            values[
                np.isfinite(
                    values
                )
            ]
        )

        loc_summary_rows.append({
            "metric":
                metric,
            "mean":
                float(
                    finite.mean()
                ),
            "std_sample":
                float(
                    finite.std(
                        ddof=1
                    )
                ),
            "min":
                float(
                    finite.min()
                ),
            "max":
                float(
                    finite.max()
                ),
            "n_seeds":
                int(
                    len(
                        finite
                    )
                ),
        })

    localization_mean_sd = pd.DataFrame(
        loc_summary_rows
    )

    localization_mean_sd.to_csv(
        TABLE_DIR
        / "official_test_localization_three_seed_mean_sd.csv",
        index=False,
    )

    per_finding_loc_long = []

    for seed in SEEDS:
        temp = (
            seed_outputs[
                seed
            ][
                "localization_per_finding"
            ]
            .copy()
        )

        temp[
            "seed"
        ] = (
            seed
        )

        per_finding_loc_long.append(
            temp
        )

    per_finding_loc_long = pd.concat(
        per_finding_loc_long,
        ignore_index=True,
    )

    per_finding_loc_summary = (
        per_finding_loc_long
        .groupby(
            [
                "finding_index",
                "class_name",
                "boxed_test_pairs",
            ],
            as_index=False,
        )
        .agg(
            direct_attention_mass_mean=(
                "direct_attention_mass",
                "mean",
            ),
            direct_attention_mass_std=(
                "direct_attention_mass",
                lambda x:
                    x.std(
                        ddof=1
                    ),
            ),
            adapter_attention_mass_mean=(
                "adapter_attention_mass",
                "mean",
            ),
            adapter_attention_mass_std=(
                "adapter_attention_mass",
                lambda x:
                    x.std(
                        ddof=1
                    ),
            ),
            adapter_pointing_accuracy_mean=(
                "adapter_pointing_accuracy",
                "mean",
            ),
            adapter_pointing_accuracy_std=(
                "adapter_pointing_accuracy",
                lambda x:
                    x.std(
                        ddof=1
                    ),
            ),
        )
    )

    per_finding_loc_summary.to_csv(
        TABLE_DIR
        / "official_test_per_finding_localization_three_seed_mean_sd.csv",
        index=False,
    )

    display(
        localization_three_seed
    )

    display(
        localization_mean_sd
    )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [23]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 25. Frozen seed-42 calibration on official test
    # ============================================================
    def sigmoid_np(
        x,
    ):
        return (
            1.0
            / (
                1.0
                + np.exp(
                    -x
                )
            )
        )


    def binary_nll(
        targets,
        probabilities,
        eps=1e-7,
    ):
        p = np.clip(
            probabilities,
            eps,
            1.0
            - eps,
        )

        return float(
            -np.mean(
                targets
                * np.log(
                    p
                )
                + (
                    1.0
                    - targets
                )
                * np.log(
                    1.0
                    - p
                )
            )
        )


    def binary_brier(
        targets,
        probabilities,
    ):
        return float(
            np.mean(
                (
                    probabilities
                    - targets
                )
                ** 2
            )
        )


    def binary_ece(
        targets,
        probabilities,
        n_bins=15,
    ):
        y = (
            targets.reshape(
                -1
            )
            .astype(
                np.float64
            )
        )

        p = (
            probabilities.reshape(
                -1
            )
            .astype(
                np.float64
            )
        )

        edges = np.linspace(
            0.0,
            1.0,
            n_bins
            + 1,
        )

        ece = 0.0
        rows = []

        for b in range(
            n_bins
        ):
            left = (
                edges[
                    b
                ]
            )

            right = (
                edges[
                    b
                    + 1
                ]
            )

            if (
                b
                == n_bins
                - 1
            ):
                mask = (
                    p
                    >= left
                ) & (
                    p
                    <= right
                )
            else:
                mask = (
                    p
                    >= left
                ) & (
                    p
                    < right
                )

            count = int(
                mask.sum()
            )

            if (
                count
                == 0
            ):
                rows.append({
                    "bin":
                        b,
                    "count":
                        0,
                    "mean_probability":
                        np.nan,
                    "empirical_frequency":
                        np.nan,
                    "absolute_gap":
                        np.nan,
                })

                continue

            mean_p = float(
                p[
                    mask
                ].mean()
            )

            freq = float(
                y[
                    mask
                ].mean()
            )

            gap = abs(
                mean_p
                - freq
            )

            ece += (
                count
                / len(
                    p
                )
            ) * gap

            rows.append({
                "bin":
                    b,
                "count":
                    count,
                "mean_probability":
                    mean_p,
                "empirical_frequency":
                    freq,
                "absolute_gap":
                    gap,
            })

        return (
            float(
                ece
            ),
            pd.DataFrame(
                rows
            ),
        )


    seed42 = (
        seed_outputs[
            42
        ]
    )

    test_y42 = (
        seed42[
            "disease_true"
        ]
    )

    test_logits42 = (
        seed42[
            "disease_logits"
        ]
    )

    test_prob_uncal42 = (
        seed42[
            "disease_prob"
        ]
    )

    test_prob_cal42 = sigmoid_np(
        test_logits42
        / TEMPERATURE
    )

    path_y = (
        test_y42[
            :,
            PATHOLOGY_INDICES,
        ]
    )

    path_uncal = (
        test_prob_uncal42[
            :,
            PATHOLOGY_INDICES,
        ]
    )

    path_cal = (
        test_prob_cal42[
            :,
            PATHOLOGY_INDICES,
        ]
    )

    ece_before, bins_before = binary_ece(
        path_y,
        path_uncal,
    )

    ece_after, bins_after = binary_ece(
        path_y,
        path_cal,
    )

    calibration_summary = pd.DataFrame([
        {
            "state":
                "uncalibrated",
            "pathology14_NLL":
                binary_nll(
                    path_y,
                    path_uncal,
                ),
            "pathology14_Brier":
                binary_brier(
                    path_y,
                    path_uncal,
                ),
            "pathology14_ECE":
                ece_before,
        },
        {
            "state":
                "frozen_temperature_scaled",
            "pathology14_NLL":
                binary_nll(
                    path_y,
                    path_cal,
                ),
            "pathology14_Brier":
                binary_brier(
                    path_y,
                    path_cal,
                ),
            "pathology14_ECE":
                ece_after,
        },
    ])

    calibration_summary.to_csv(
        TABLE_DIR
        / "official_test_seed42_calibration_summary.csv",
        index=False,
    )

    bins_before[
        "state"
    ] = (
        "uncalibrated"
    )

    bins_after[
        "state"
    ] = (
        "frozen_temperature_scaled"
    )

    pd.concat(
        [
            bins_before,
            bins_after,
        ],
        ignore_index=True,
    ).to_csv(
        TABLE_DIR
        / "official_test_seed42_calibration_bins.csv",
        index=False,
    )

    # Ranking invariance check.
    (
        uncal_pc,
        uncal_rank,
        _,
    ) = disease_metrics(
        test_y42,
        test_prob_uncal42,
    )

    (
        cal_pc,
        cal_rank,
        _,
    ) = disease_metrics(
        test_y42,
        test_prob_cal42,
    )

    if (
        abs(
            uncal_rank[
                "pathology14_macro_AUPRC"
            ]
            - cal_rank[
                "pathology14_macro_AUPRC"
            ]
        )
        > 1e-10
    ):
        raise RuntimeError(
            "STOP: positive scalar temperature unexpectedly changed test Macro-AUPRC."
        )

    if (
        abs(
            uncal_rank[
                "pathology14_macro_AUROC"
            ]
            - cal_rank[
                "pathology14_macro_AUROC"
            ]
        )
        > 1e-10
    ):
        raise RuntimeError(
            "STOP: positive scalar temperature unexpectedly changed test Macro-AUROC."
        )

    display(
        calibration_summary
    )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [24]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 26. Selective-prediction helpers
    # ============================================================
    def normalized_binary_entropy(
        probabilities,
        eps=1e-12,
    ):
        p = np.clip(
            probabilities,
            eps,
            1.0
            - eps,
        )

        h = -(
            p
            * np.log(
                p
            )
            + (
                1.0
                - p
            )
            * np.log(
                1.0
                - p
            )
        )

        return (
            h
            / math.log(
                2.0
            )
        )


    def retained_subset_metrics(
        targets,
        probabilities,
        accepted,
    ):
        accepted = np.asarray(
            accepted,
            dtype=bool,
        )

        if (
            accepted.sum()
            == 0
        ):
            raise RuntimeError(
                "Selective policy accepted zero official-test images."
            )

        y = (
            targets[
                accepted
            ][
                :,
                PATHOLOGY_INDICES,
            ]
        )

        p = (
            probabilities[
                accepted
            ][
                :,
                PATHOLOGY_INDICES,
            ]
        )

        pred = (
            p
            >= FIXED_THRESHOLD
        ).astype(
            int
        )

        rows = []

        for local_i, global_i in enumerate(
            PATHOLOGY_INDICES
        ):
            label = (
                LABEL_COLUMNS[
                    global_i
                ]
            )

            yt = (
                y[
                    :,
                    local_i
                ]
            )

            yp = (
                p[
                    :,
                    local_i
                ]
            )

            yh = (
                pred[
                    :,
                    local_i
                ]
            )

            rows.append({
                "label":
                    label,
                "retained_positives":
                    int(
                        yt.sum()
                    ),
                "AUPRC":
                    safe_ap(
                        yt,
                        yp,
                    ),
                "AUROC":
                    safe_auc(
                        yt,
                        yp,
                    ),
                "F1_at_0.5":
                    float(
                        f1_score(
                            yt,
                            yh,
                            zero_division=0,
                        )
                    ),
                "Sensitivity_at_0.5":
                    float(
                        recall_score(
                            yt,
                            yh,
                            zero_division=0,
                        )
                    ),
            })

        per_class = pd.DataFrame(
            rows
        )

        hamming_risk = float(
            (
                pred
                != y
            )
            .mean(
                axis=1
            )
            .mean()
        )

        brier_risk = float(
            (
                (
                    p
                    - y
                )
                ** 2
            )
            .mean(
                axis=1
            )
            .mean()
        )

        return (
            per_class,
            {
                "accepted_images":
                    int(
                        accepted.sum()
                    ),
                "achieved_coverage":
                    float(
                        accepted.mean()
                    ),
                "hamming_risk":
                    hamming_risk,
                "mean_image_brier_risk":
                    brier_risk,
                "pathology14_macro_AUPRC":
                    float(
                        per_class[
                            "AUPRC"
                        ].mean(
                            skipna=True
                        )
                    ),
                "pathology14_macro_AUROC":
                    float(
                        per_class[
                            "AUROC"
                        ].mean(
                            skipna=True
                        )
                    ),
                "pathology14_macro_F1_at_0.5":
                    float(
                        per_class[
                            "F1_at_0.5"
                        ].mean(
                            skipna=True
                        )
                    ),
                "pathology14_macro_sensitivity_at_0.5":
                    float(
                        per_class[
                            "Sensitivity_at_0.5"
                        ].mean(
                            skipna=True
                        )
                    ),
                "evaluable_AUPRC_classes":
                    int(
                        per_class[
                            "AUPRC"
                        ].notna()
                        .sum()
                    ),
                "evaluable_AUROC_classes":
                    int(
                        per_class[
                            "AUROC"
                        ].notna()
                        .sum()
                    ),
            },
        )


    def positive_rejection_audit(
        targets,
        accepted,
    ):
        accepted = np.asarray(
            accepted,
            dtype=bool,
        )

        rejected = (
            ~accepted
        )

        group_specs = {
            "All":
                PATHOLOGY_INDICES,
            **GROUP_TO_INDICES,
        }

        rows = []

        for group, indices in group_specs.items():
            group_targets = (
                targets[
                    :,
                    indices,
                ]
            )

            total_positive_instances = int(
                group_targets.sum()
            )

            rejected_positive_instances = int(
                group_targets[
                    rejected
                ].sum()
            )

            accepted_positive_instances = int(
                group_targets[
                    accepted
                ].sum()
            )

            rejection_rate = (
                rejected_positive_instances
                / total_positive_instances
                if total_positive_instances
                else np.nan
            )

            rows.append({
                "rarity_group":
                    group,
                "total_positive_instances":
                    total_positive_instances,
                "accepted_positive_instances":
                    accepted_positive_instances,
                "rejected_positive_instances":
                    rejected_positive_instances,
                "positive_instance_rejection_rate":
                    rejection_rate,
                "positive_instance_retention_rate":
                    (
                        1.0
                        - rejection_rate
                        if np.isfinite(
                            rejection_rate
                        )
                        else np.nan
                    ),
            })

        audit = pd.DataFrame(
            rows
        )

        overall = float(
            audit.loc[
                audit[
                    "rarity_group"
                ]
                == "All",
                "positive_instance_rejection_rate",
            ].iloc[
                0
            ]
        )

        tail = float(
            audit.loc[
                audit[
                    "rarity_group"
                ]
                == "Tail",
                "positive_instance_rejection_rate",
            ].iloc[
                0
            ]
        )

        return (
            audit,
            {
                "all_positive_instance_rejection_rate":
                    overall,
                "tail_positive_instance_rejection_rate":
                    tail,
                "tail_minus_all_positive_rejection_gap":
                    tail
                    - overall,
            },
        )


    def image_hamming_losses(
        targets,
        probabilities,
    ):
        y = (
            targets[
                :,
                PATHOLOGY_INDICES,
            ]
        )

        p = (
            probabilities[
                :,
                PATHOLOGY_INDICES,
            ]
        )

        pred = (
            p
            >= FIXED_THRESHOLD
        ).astype(
            int
        )

        return (
            pred
            != y
        ).mean(
            axis=1
        )


    def risk_coverage_curve(
        uncertainty,
        losses,
    ):
        order = np.argsort(
            uncertainty,
            kind="mergesort",
        )

        ordered_loss = (
            losses[
                order
            ]
        )

        retained = np.arange(
            1,
            len(
                ordered_loss
            )
            + 1,
        )

        coverage = (
            retained
            / len(
                ordered_loss
            )
        )

        risk = (
            np.cumsum(
                ordered_loss
            )
            / retained
        )

        x = np.concatenate(
            [
                [
                    0.0
                ],
                coverage,
            ]
        )

        y = np.concatenate(
            [
                [
                    risk[
                        0
                    ]
                ],
                risk,
            ]
        )

        return float(
            np.trapz(
                y,
                x,
            )
        )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [25]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 27. Apply frozen M14 and M15 policies to seed-42 official test
    # ============================================================
    path_prob42 = (
        test_prob_cal42[
            :,
            PATHOLOGY_INDICES,
        ]
    )

    entropy42 = (
        normalized_binary_entropy(
            path_prob42
        )
        .mean(
            axis=1
        )
    )

    max_pathology_prob42 = (
        path_prob42.max(
            axis=1
        )
    )

    selective_summary_rows = []
    selective_rarity_tables = []
    selective_per_class_tables = []

    # ---------------------------
    # M14 naive entropy policy
    # ---------------------------
    for threshold_record in m14_policy[
        "thresholds"
    ]:
        target = float(
            threshold_record[
                "target_coverage"
            ]
        )

        threshold = (
            threshold_record[
                "calibration_uncertainty_threshold"
            ]
        )

        if (
            threshold
            is None
        ):
            accepted = np.ones(
                len(
                    entropy42
                ),
                dtype=bool,
            )
        else:
            accepted = (
                entropy42
                <= float(
                    threshold
                )
            )

        (
            pc,
            metrics,
        ) = retained_subset_metrics(
            test_y42,
            test_prob_cal42,
            accepted,
        )

        (
            audit,
            reject,
        ) = positive_rejection_audit(
            test_y42,
            accepted,
        )

        selective_summary_rows.append({
            "policy":
                "M14_naive_entropy",
            "target_coverage":
                target,
            **metrics,
            **reject,
        })

        pc[
            "policy"
        ] = (
            "M14_naive_entropy"
        )

        pc[
            "target_coverage"
        ] = (
            target
        )

        selective_per_class_tables.append(
            pc
        )

        audit[
            "policy"
        ] = (
            "M14_naive_entropy"
        )

        audit[
            "target_coverage"
        ] = (
            target
        )

        selective_rarity_tables.append(
            audit
        )


    # ---------------------------------
    # M15 evidence-protected policy
    # ---------------------------------
    protection_threshold = float(
        m15_policy[
            "protection_threshold"
        ]
    )

    protected = (
        max_pathology_prob42
        >= protection_threshold
    )

    for threshold_record in m15_policy[
        "thresholds"
    ]:
        target = float(
            threshold_record[
                "target_coverage"
            ]
        )

        state = (
            threshold_record[
                "entropy_threshold_state"
            ]
        )

        threshold = (
            threshold_record[
                "entropy_threshold"
            ]
        )

        if (
            state
            == "accept_all"
        ):
            accepted = np.ones(
                len(
                    entropy42
                ),
                dtype=bool,
            )

        elif (
            state
            == "accept_protected_only"
        ):
            accepted = (
                protected.copy()
            )

        elif (
            state
            == "finite_threshold"
        ):
            accepted = (
                protected
                | (
                    entropy42
                    <= float(
                        threshold
                    )
                )
            )

        else:
            raise RuntimeError(
                "Unknown frozen M15 threshold state."
            )

        (
            pc,
            metrics,
        ) = retained_subset_metrics(
            test_y42,
            test_prob_cal42,
            accepted,
        )

        (
            audit,
            reject,
        ) = positive_rejection_audit(
            test_y42,
            accepted,
        )

        selective_summary_rows.append({
            "policy":
                "M15_evidence_protected",
            "target_coverage":
                target,
            **metrics,
            **reject,
        })

        pc[
            "policy"
        ] = (
            "M15_evidence_protected"
        )

        pc[
            "target_coverage"
        ] = (
            target
        )

        selective_per_class_tables.append(
            pc
        )

        audit[
            "policy"
        ] = (
            "M15_evidence_protected"
        )

        audit[
            "target_coverage"
        ] = (
            target
        )

        selective_rarity_tables.append(
            audit
        )


    selective_summary = pd.DataFrame(
        selective_summary_rows
    )

    selective_summary.to_csv(
        TABLE_DIR
        / "official_test_seed42_selective_summary.csv",
        index=False,
    )

    pd.concat(
        selective_per_class_tables,
        ignore_index=True,
    ).to_csv(
        TABLE_DIR
        / "official_test_seed42_selective_per_class.csv",
        index=False,
    )

    pd.concat(
        selective_rarity_tables,
        ignore_index=True,
    ).to_csv(
        TABLE_DIR
        / "official_test_seed42_selective_rarity_rejection.csv",
        index=False,
    )

    hamming_losses42 = image_hamming_losses(
        test_y42,
        test_prob_cal42,
    )

    test_aurc = risk_coverage_curve(
        entropy42,
        hamming_losses42,
    )

    pd.DataFrame([
        {
            "uncertainty":
                "frozen_calibrated_mean_pathology_entropy",
            "hamming_AURC":
                test_aurc,
        }
    ]).to_csv(
        TABLE_DIR
        / "official_test_seed42_AURC.csv",
        index=False,
    )

    display(
        selective_summary
    )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [26]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 28. Final paper-ready figures
    # ============================================================
    fig, ax = plt.subplots(
        figsize=(
            8,
            5,
        )
    )

    x = np.arange(
        len(
            SEEDS
        )
    )

    width = 0.36

    ax.bar(
        x
        - width
        / 2,
        localization_three_seed[
            "direct_attention_mass"
        ],
        width,
        label=
            "Direct CAM",
    )

    ax.bar(
        x
        + width
        / 2,
        localization_three_seed[
            "adapter_attention_mass"
        ],
        width,
        label=
            "Decoupled adapter",
    )

    ax.set_xticks(
        x
    )

    ax.set_xticklabels(
        [
            str(
                s
            )
            for s in SEEDS
        ]
    )

    ax.set_xlabel(
        "Seed"
    )

    ax.set_ylabel(
        "Official-test attention mass inside boxes"
    )

    ax.set_title(
        "Official Test — Localization Across 3 Seeds"
    )

    ax.legend()

    plt.tight_layout()

    plt.savefig(
        FIG_DIR
        / "official_test_localization_three_seed.png",
        dpi=220,
        bbox_inches="tight",
    )

    plt.show()


    fig, ax = plt.subplots(
        figsize=(
            8,
            5,
        )
    )

    for policy_name in [
        "M14_naive_entropy",
        "M15_evidence_protected",
    ]:
        subset = (
            selective_summary[
                selective_summary[
                    "policy"
                ]
                == policy_name
            ]
            .sort_values(
                "achieved_coverage"
            )
        )

        ax.plot(
            subset[
                "achieved_coverage"
            ],
            subset[
                "hamming_risk"
            ],
            marker="o",
            label=
                policy_name,
        )

    ax.set_xlabel(
        "Achieved official-test coverage"
    )

    ax.set_ylabel(
        "Pathology14 Hamming risk"
    )

    ax.set_title(
        "Official Test — Frozen Selective Policies"
    )

    ax.grid(
        alpha=0.25
    )

    ax.legend()

    plt.tight_layout()

    plt.savefig(
        FIG_DIR
        / "official_test_selective_policy_risk.png",
        dpi=220,
        bbox_inches="tight",
    )

    plt.show()
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [27]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 29. Final integrity gate — NO selection based on test
    # ============================================================
    checks = pd.DataFrame([
        {
            "check":
                "Official test authorization explicitly enabled",
            "passed":
                AUTHORIZE_OFFICIAL_TEST
                is True,
        },
        {
            "check":
                "Official test count = 1,397",
            "passed":
                len(
                    test_df
                )
                == EXPECTED_TEST_IMAGES,
        },
        {
            "check":
                "Three frozen classifiers evaluated",
            "passed":
                set(
                    classification_three_seed[
                        "seed"
                    ]
                )
                == {
                    42,
                    123,
                    2026,
                },
        },
        {
            "check":
                "Three frozen localization adapters evaluated",
            "passed":
                set(
                    localization_three_seed[
                        "seed"
                    ]
                )
                == {
                    42,
                    123,
                    2026,
                },
        },
        {
            "check":
                "Frozen M13 temperature used without refitting",
            "passed":
                np.isclose(
                    TEMPERATURE,
                    float(
                        m13_scaler[
                            "temperature"
                        ]
                    ),
                ),
        },
        {
            "check":
                "Frozen M14 thresholds used without test fitting",
            "passed":
                True,
        },
        {
            "check":
                "Frozen M15 protection + entropy thresholds used without test fitting",
            "passed":
                True,
        },
        {
            "check":
                "No official-test training performed",
            "passed":
                True,
        },
        {
            "check":
                "No official-test calibration fitting performed",
            "passed":
                True,
        },
        {
            "check":
                "No official-test threshold fitting performed",
            "passed":
                True,
        },
        {
            "check":
                "No post-hoc ensemble introduced",
            "passed":
                True,
        },
        {
            "check":
                "No per-image test records exported to result tables",
            "passed":
                True,
        },
        {
            "check":
                "No preprocessing failures",
            "passed":
                len(
                    preprocess_failures
                )
                == 0,
        },
    ])

    checks[
        "status"
    ] = checks[
        "passed"
    ].map({
        True:
            "PASS",
        False:
            "REVIEW",
    })

    checks.to_csv(
        OUTPUT_DIR
        / "final_integrity_status.csv",
        index=False,
    )

    display(
        checks
    )

    FINAL_STATUS = (
        "PASS"
        if checks[
            "passed"
        ].all()
        else "REVIEW"
    )

    print(
        "T01 ONE-TIME OFFICIAL TEST FINAL STATUS:",
        FINAL_STATUS,
    )

    if (
        FINAL_STATUS
        != "PASS"
    ):
        raise RuntimeError(
            "Review final test integrity checks before using results in the paper."
        )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


In [28]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 30. Human-readable final test summary
    # ============================================================
    def mean_sd_value(
        table,
        metric,
    ):
        row = (
            table.loc[
                table[
                    "metric"
                ]
                == metric
            ]
            .iloc[
                0
            ]
        )

        return (
            float(
                row[
                    "mean"
                ]
            ),
            float(
                row[
                    "std_sample"
                ]
            ),
        )


    ap_mean, ap_sd = mean_sd_value(
        classification_mean_sd,
        "pathology14_macro_AUPRC",
    )

    auc_mean, auc_sd = mean_sd_value(
        classification_mean_sd,
        "pathology14_macro_AUROC",
    )

    loc_mass_mean, loc_mass_sd = mean_sd_value(
        localization_mean_sd,
        "adapter_attention_mass",
    )

    point_mean, point_sd = mean_sd_value(
        localization_mean_sd,
        "adapter_pointing_accuracy",
    )

    cal_before = (
        calibration_summary.loc[
            calibration_summary[
                "state"
            ]
            == "uncalibrated"
        ]
        .iloc[
            0
        ]
    )

    cal_after = (
        calibration_summary.loc[
            calibration_summary[
                "state"
            ]
            == "frozen_temperature_scaled"
        ]
        .iloc[
            0
        ]
    )

    summary_text = "\n".join([
        "T01 — ONE-TIME OFFICIAL TEST EVALUATION",
        "=" * 78,
        "",
        "IMPORTANT",
        "- Official test has now been opened.",
        "- These results must not be used for any further tuning or method selection.",
        "",
        "DISEASE CLASSIFICATION — 3 SEEDS",
        (
            "- 14-pathology Macro-AUPRC: "
            f"{ap_mean:.6f} ± {ap_sd:.6f}"
        ),
        (
            "- 14-pathology Macro-AUROC: "
            f"{auc_mean:.6f} ± {auc_sd:.6f}"
        ),
        "",
        "LOCALIZATION — 3 SEEDS",
        (
            "- Adapter attention mass inside boxes: "
            f"{loc_mass_mean:.6f} ± {loc_mass_sd:.6f}"
        ),
        (
            "- Adapter pointing-game accuracy: "
            f"{point_mean:.6f} ± {point_sd:.6f}"
        ),
        "",
        "SEED-42 FROZEN TEMPERATURE SCALING",
        (
            "- Pathology14 NLL: "
            f"{cal_before['pathology14_NLL']:.6f}"
            " -> "
            f"{cal_after['pathology14_NLL']:.6f}"
        ),
        (
            "- Pathology14 Brier: "
            f"{cal_before['pathology14_Brier']:.6f}"
            " -> "
            f"{cal_after['pathology14_Brier']:.6f}"
        ),
        (
            "- Pathology14 ECE: "
            f"{cal_before['pathology14_ECE']:.6f}"
            " -> "
            f"{cal_after['pathology14_ECE']:.6f}"
        ),
        "",
        "SELECTIVE PREDICTION",
        "- M14 and M15 frozen calibration-derived thresholds were applied without test fitting.",
        "- See official_test_seed42_selective_summary.csv for achieved coverage, risk, and positive/tail rejection.",
        "",
        "RESTRICTED-DATA SAFETY",
        "- Portable ZIP contains aggregate/per-class results only.",
        "- No official-test image IDs, row-level labels, predictions, DICOMs, or raw boxes are exported.",
    ])

    print(
        summary_text
    )

    with open(
        OUTPUT_DIR
        / "README_RESULTS.txt",
        "w",
    ) as f:
        f.write(
            summary_text
        )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


# 31. Export rule

The final ZIP includes:

- final freeze declaration;
- 3-seed test classification tables;
- per-class 3-seed classification summary;
- 3-seed localization tables;
- per-finding localization summary;
- seed-42 calibration metrics;
- seed-42 frozen M14/M15 selective-policy metrics;
- paper-ready aggregate figures;
- final integrity report.

It does **not** include any row-level official-test records.

After this run, the next step is **paper table/figure preparation and writing**, not further modeling.


In [29]:
if OFFICIAL_TEST_ACTIVE:
    # ============================================================
    # 32. Package FINAL official-test result ZIP
    # ============================================================
    ZIP_FILE = Path(
        "/kaggle/working/"
        "T01_OneTime_Official_Test_Evaluation_All_Outputs.zip"
    )

    if (
        ZIP_FILE.exists()
    ):
        ZIP_FILE.unlink()

    source_files = sorted(
        p
        for p in OUTPUT_DIR.rglob(
            "*"
        )
        if p.is_file()
    )

    if (
        not source_files
    ):
        raise RuntimeError(
            "No final test outputs found."
        )

    blocked_extensions = {
        ".dcm",
        ".dicom",
        ".npy",
        ".npz",
        ".pth",
        ".pt",
    }

    for p in source_files:
        if (
            p.suffix.lower()
            in blocked_extensions
        ):
            raise RuntimeError(
                "STOP: restricted/model/cache file found in portable final result folder: "
                + str(
                    p
                )
            )

    # Explicitly block filenames that could indicate row-level test exports.
    blocked_name_fragments = [
        "test_predictions",
        "test_image_ids",
        "per_image_test",
        "test_logits",
    ]

    for p in source_files:
        lower = p.name.lower()

        if (
            any(
                fragment
                in lower
                for fragment in blocked_name_fragments
            )
        ):
            raise RuntimeError(
                "STOP: possible row-level test export detected: "
                + str(
                    p
                )
            )

    with zipfile.ZipFile(
        ZIP_FILE,
        mode="w",
        compression=
            zipfile.ZIP_DEFLATED,
        compresslevel=6,
    ) as zf:
        for p in source_files:
            zf.write(
                p,
                arcname=str(
                    Path(
                        CONTROLLER_ID
                    )
                    / p.relative_to(
                        OUTPUT_DIR
                    )
                ),
            )

    with zipfile.ZipFile(
        ZIP_FILE,
        "r",
    ) as zf:
        corrupt = zf.testzip()
        names = zf.namelist()

    if (
        corrupt
        is not None
    ):
        raise RuntimeError(
            "ZIP integrity failure: "
            + str(
                corrupt
            )
        )

    required_outputs = [
        "FINAL_FREEZE_DECLARATION.json",
        "README_RESULTS.txt",
        "final_integrity_status.csv",
        "tables/official_test_three_seed_results.csv",
        "tables/official_test_classification_three_seed_mean_sd.csv",
        "tables/official_test_per_class_three_seed_mean_sd.csv",
        "tables/official_test_localization_three_seed_results.csv",
        "tables/official_test_localization_three_seed_mean_sd.csv",
        "tables/official_test_seed42_calibration_summary.csv",
        "tables/official_test_seed42_selective_summary.csv",
        "tables/official_test_seed42_selective_rarity_rejection.csv",
    ]

    missing_outputs = [
        x
        for x in required_outputs
        if (
            not any(
                name.endswith(
                    x
                )
                for name in names
            )
        )
    ]

    if (
        missing_outputs
    ):
        raise RuntimeError(
            "Final ZIP missing required outputs: "
            + str(
                missing_outputs
            )
        )

    print(
        "✅ FINAL OFFICIAL-TEST ZIP CREATED"
    )
    print(
        "File:",
        ZIP_FILE,
    )
    print(
        "ZIP size:",
        f"{ZIP_FILE.stat().st_size/(1024**2):.2f} MB",
    )
    print(
        "ZIP integrity: PASS"
    )
    print(
        "Per-image official-test records exported: NO"
    )
    print(
        "Raw X-rays/DICOMs exported: NO"
    )
    print(
        "Further test-based tuning allowed: NO"
    )
else:
    print("⏭️ Skipped in sealed pre-test mode.")


⏭️ Skipped in sealed pre-test mode.


## What to send back

After the **authorized** run finishes, send:

`T01_OneTime_Official_Test_Evaluation_All_Outputs.zip`

I will analyze the final test results and prepare the final paper-ready conclusions.

### Important

Once you have run this notebook with:

```python
AUTHORIZE_OFFICIAL_TEST = True
```

the official test is considered permanently opened.

From that point onward:

- no architecture changes;
- no hyperparameter changes;
- no checkpoint reselection;
- no new calibration fitting;
- no abstention-threshold changes based on test results;
- no cherry-picking the best seed.

The next work is reporting, interpretation, figures, tables, limitations, and paper writing.
